# gpt-oss-20b QLoRA fine-tune (Colab A100)

Runtime: **A100 GPU**, **High-RAM on**. Run the cells top to bottom.

Training data: a file called `train.jsonl`, one example per line, in chat format:

```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

The `system` message is optional. Upload the file in step 4.

## 1. Install

In [ ]:
%pip install -q unsloth
# If this cell or the model load below fails with a version error, copy the
# install cell from Unsloth's official gpt-oss-20b Colab notebook instead.

## 2. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > A100 GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"{gpu.name}: {gpu.total_memory / 1024**3:.0f} GB VRAM")

## 3. Load the model in 4-bit and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096  # longest example (in tokens) the model will see; lower it if you run out of memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

## 4. Upload and prepare the dataset

Skip the upload cell if `train.jsonl` is already in the Files panel.

In [ ]:
import os

if not os.path.exists("train.jsonl"):
    from google.colab import files
    files.upload()  # pick train.jsonl

assert os.path.exists("train.jsonl"), "train.jsonl not found"

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl", split="train")

def to_text(batch):
    # Renders each conversation in gpt-oss's harmony chat format
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]}

dataset = dataset.map(to_text, batched=True)

# Report how many examples are longer than MAX_SEQ_LENGTH (those get cut off)
lengths = [len(tokenizer(t).input_ids) for t in dataset["text"]]
too_long = sum(n > MAX_SEQ_LENGTH for n in lengths)
print(f"{len(dataset)} examples | longest: {max(lengths)} tokens | over the limit: {too_long}")
print("\n--- first example as the model sees it ---\n")
print(dataset[0]["text"])

In [ ]:
# Hold out 5% for evaluation once there are enough examples
if len(dataset) >= 100:
    split = dataset.train_test_split(test_size=0.05, seed=3407)
    train_ds, eval_ds = split["train"], split["test"]
else:
    train_ds, eval_ds = dataset, None
print(f"train: {len(train_ds)} | eval: {len(eval_ds) if eval_ds else 0}")

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,   # drop to 2 or 1 on an out-of-memory error
        gradient_accumulation_steps=2,   # raise this when you lower the batch size
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy="epoch" if eval_ds is not None else "no",
        save_strategy="epoch",
        output_dir="outputs",
        seed=3407,
        report_to="none",
    ),
)

In [ ]:
stats = trainer.train()
print(f"Done in {stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Peak VRAM: {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB")

## 6. Try the fine-tuned model

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "PUT A TEST PROMPT HERE (same style as your training prompts)"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="low",
).to("cuda")

_ = model.generate(**inputs, max_new_tokens=512, streamer=TextStreamer(tokenizer))

## 7. Save the adapter to Google Drive

Colab deletes local files when the session ends, so do this before you disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/gpt-oss-20b-client-brain-lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

### Loading the adapter again later

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/gpt-oss-20b-client-brain-lora",
    max_seq_length=4096,
    load_in_4bit=True,
)
```